In [4]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ==============================================================================
# 关键：导入您需要的库
# ==============================================================================
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from medpy.metric.binary import hd95 # <--- 新增: 导入hd95计算函数

# ==============================================================================
# --- 1. 配置 (您需要在这里指定要评估哪个模型) ---
# ==============================================================================
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/CVC_bLoss2.5_adapter_run1/checkpoints/checkpoint.pt"
HYDRA_OVERRIDES = [
    "model.image_encoder.trunk._target_=sam2.modeling.backbones.hieradet_adapterv1.Hiera",
    "+model.image_encoder.trunk.use_adapter=True",
    "+model.use_adapter=True",
]
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/CVC-ClinicDB_for_SAM2"
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets", "val.txt")

# ==============================================================================
# --- 2. 指标计算函数 (已修改，增加HD95) ---
# ==============================================================================
def calculate_metrics(gt_mask, pred_mask):
    """
    计算 Dice, IoU, 和 HD95.
    """
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0
    
    # Dice 和 IoU
    if not np.any(gt_mask_bool) and not np.any(pred_mask_bool):
        # 如果GT和预测都是空的，说明完美匹配
        dice, iou = 1.0, 1.0
    elif not np.any(gt_mask_bool) or not np.any(pred_mask_bool):
        # 如果其中一个是空的，说明完全不匹配
        dice, iou = 0.0, 0.0
    else:
        tp = np.logical_and(pred_mask_bool, gt_mask_bool).sum()
        fp = np.logical_and(pred_mask_bool, ~gt_mask_bool).sum()
        fn = np.logical_and(~pred_mask_bool, gt_mask_bool).sum()
        dice = (2. * tp) / (2 * tp + fp + fn + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
    
    # HD95
    hd95_score = np.nan # 默认为NaN
    if np.any(gt_mask_bool) and np.any(pred_mask_bool):
        try:
            hd95_score = hd95(pred_mask_bool, gt_mask_bool)
        except RuntimeError:
            # 在某些极端情况下medpy会失败，忽略这个样本的hd95
            pass
            
    return dice, iou, hd95_score


# ==============================================================================
# --- 3. 主评估流程 ---
# ==============================================================================
def main():
    # --- 模型加载 ---
    print("Loading SAM2 model in Adapter mode...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model = build_sam2(
        SAM2_CFG_NAME, 
        SAM2_CHECKPOINT_PATH, 
        device=device,
        hydra_overrides_extra=HYDRA_OVERRIDES
    )
    predictor = SAM2ImagePredictor(model)
    print(f"Model and Predictor loaded on {device}.")

    # --- 数据集文件读取 ---
    if not os.path.exists(TEST_SET_FILE):
        raise FileNotFoundError(f"Test set file not found: {TEST_SET_FILE}")

    with open(TEST_SET_FILE, 'r') as f:
        test_sample_names = [line.strip() for line in f.readlines()]
    
    print(f"\nFound {len(test_sample_names)} samples in the test set. Starting evaluation...")
    
    results = []
    images_dir = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
    annotations_dir = os.path.join(SPLIT_DATASET_ROOT, "Annotations")

    for sample_name in tqdm(test_sample_names, desc="Evaluating Test Set"):
        image_path = os.path.join(images_dir, sample_name, "00000.jpg")
        mask_path = os.path.join(annotations_dir, sample_name, "00000.png")
        
        if not os.path.exists(image_path) or not os.path.exists(mask_path):
            continue

        image = cv2.imread(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        predictor.set_image(image_rgb)
        contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        box_prompt = np.array([[x, y, x + w, y + h]])

        masks, _, _ = predictor.predict(box=box_prompt, multimask_output=False)
        pred_mask = (masks[0] * 255).astype(np.uint8)
        
        # 计算所有三个指标
        dice, iou, hd95_score = calculate_metrics(gt_mask, pred_mask)
        
        # <--- 新增: 从样本名解析类别 ---
        category = 'benign' if 'benign' in sample_name else 'malignant' if 'malignant' in sample_name else 'polyp'

        results.append({
            "sample_name": sample_name,
            "category": category,
            "dice": dice, 
            "iou": iou, 
            "hd95": hd95_score
        })

        predictor.reset_predictor()

    # ==============================================================================
    # --- 4. 结果汇总与展示 (已完全替换为您的新格式) ---
    # ==============================================================================
    if results:
        df = pd.DataFrame(results)
        
        # 按类别计算平均值和标准差
        agg_funcs = {
            'dice': ['mean', 'std'],
            'iou': ['mean', 'std'],
            'hd95': ['mean', 'std'],
            'sample_name': ['count']
        }
        category_summary = df.groupby('category').agg(agg_funcs).reset_index()
        
        # 扁平化多级列名
        category_summary.columns = ['_'.join(col).strip() for col in category_summary.columns.values]
        category_summary.rename(columns={'category_': 'Category', 'sample_name_count': 'Count'}, inplace=True)

        # 计算并添加总体平均指标
        overall_metrics = {
            'Category': 'Overall',
            'dice_mean': df['dice'].mean(),
            'dice_std': df['dice'].std(),
            'iou_mean': df['iou'].mean(),
            'iou_std': df['iou'].std(),
            'hd95_mean': df['hd95'].mean(), # .mean() 会自动忽略NaN
            'hd95_std': df['hd95'].std(),
            'Count': len(df)
        }
        
        # 将总体指标作为新行添加到汇总DataFrame中
        overall_df = pd.DataFrame([overall_metrics])
        final_summary = pd.concat([category_summary, overall_df], ignore_index=True)

        print("\n" + "="*85)
        print("--- Evaluation Summary on Test Set ---")
        print("="*85)
        
        # 格式化浮点数，让输出更美观
        formatters = {
            'dice_mean': "{:.4f}".format, 'dice_std': "{:.4f}".format,
            'iou_mean': "{:.4f}".format, 'iou_std': "{:.4f}".format,
            'hd95_mean': "{:.2f}".format, 'hd95_std': "{:.2f}".format,
        }
        print(final_summary.to_string(index=False, formatters=formatters))
        print("="*85)

    else:
        print("No images were processed. Please check paths and file structure.")

if __name__ == '__main__':
    main()

Loading SAM2 model in Adapter mode...


RuntimeError: Error(s) in loading state_dict for SAM2Base:
	Missing key(s) in state_dict: "image_encoder.trunk.blocks.0.adapter.down_project.weight", "image_encoder.trunk.blocks.0.adapter.down_project.bias", "image_encoder.trunk.blocks.0.adapter.up_project.weight", "image_encoder.trunk.blocks.0.adapter.up_project.bias", "image_encoder.trunk.blocks.1.adapter.down_project.weight", "image_encoder.trunk.blocks.1.adapter.down_project.bias", "image_encoder.trunk.blocks.1.adapter.up_project.weight", "image_encoder.trunk.blocks.1.adapter.up_project.bias", "image_encoder.trunk.blocks.2.adapter.down_project.weight", "image_encoder.trunk.blocks.2.adapter.down_project.bias", "image_encoder.trunk.blocks.2.adapter.up_project.weight", "image_encoder.trunk.blocks.2.adapter.up_project.bias", "image_encoder.trunk.blocks.3.adapter.down_project.weight", "image_encoder.trunk.blocks.3.adapter.down_project.bias", "image_encoder.trunk.blocks.3.adapter.up_project.weight", "image_encoder.trunk.blocks.3.adapter.